# 📥 MultiCaRe Dataset — Full Download & Single Merged Table

Downloads **every file** of the **[MultiCaRe](https://zenodo.org/records/20416562)**
clinical case dataset (98,000+ de-identified clinical cases, 139,000+ medical
images across all 9 image shards, ~3.3 GB total) and combines everything —
captions, image labels, clinical case text, abstracts, and article metadata —
into **one merged table**, one row per image.

DOI: `10.5281/zenodo.20416562` · [Zenodo record](https://zenodo.org/records/20416562) ·
[GitHub](https://github.com/mauro-nievoff/MultiCaRe_Dataset)

This notebook only fetches and organizes the data — no models, no VQA,
captioning, or report generation.


## Setup

In [1]:
%pip install -q requests pandas pyarrow tqdm pillow


In [2]:
import zipfile
from pathlib import Path

import requests
import pandas as pd
from tqdm.auto import tqdm


## Configuration

The full dataset is ~3 GB of metadata tables plus ~3 GB of images, split
across 9 image shards (`PMC1.zip` … `PMC9.zip`, ranging from 56 MB to 918 MB,
~3 GB total). This notebook downloads **all of it** — every metadata table
and every image shard — and extracts everything to disk before building one
merged table.

Set `DOWNLOAD_IMAGES = False` below only if you want to skip the ~3 GB of
image files and keep just the metadata/captions/case text.


In [3]:
DATA_DIR = Path("multicare_data")
IMG_DIR = DATA_DIR / "images"
DATA_DIR.mkdir(exist_ok=True, parents=True)

ZENODO_RECORD = "https://zenodo.org/records/20416562/files"

DOWNLOAD_IMAGES = True
SHARDS = ["PMC1", "PMC2", "PMC3", "PMC4", "PMC5", "PMC6", "PMC7", "PMC8", "PMC9"]  # all image shards

ALL_SHARDS_INFO = {
    "PMC1": "917.6 MB", "PMC2": "56.4 MB", "PMC3": "306.9 MB", "PMC4": "323.3 MB",
    "PMC5": "263.7 MB", "PMC6": "275.0 MB", "PMC7": "224.3 MB", "PMC8": "237.9 MB",
    "PMC9": "56.3 MB",
}
for s in SHARDS:
    print(f"{s}.zip -> {ALL_SHARDS_INFO[s]}")
print("\nTotal image download size: ~3.0 GB (plus ~300 MB of metadata tables)")


PMC1.zip -> 917.6 MB
PMC2.zip -> 56.4 MB
PMC3.zip -> 306.9 MB
PMC4.zip -> 323.3 MB
PMC5.zip -> 263.7 MB
PMC6.zip -> 275.0 MB
PMC7.zip -> 224.3 MB
PMC8.zip -> 237.9 MB
PMC9.zip -> 56.3 MB

Total image download size: ~3.0 GB (plus ~300 MB of metadata tables)


## Download the metadata tables

In [4]:
def download(url: str, dest: Path, chunk: int = 1 << 20) -> Path:
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"already have {dest.name}")
        return dest
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, desc=dest.name
        ) as pbar:
            for c in r.iter_content(chunk_size=chunk):
                f.write(c)
                pbar.update(len(c))
    return dest


META_FILES = [
    "captions_and_labels.csv",   # per-image caption + visual labels (image_type, image_subtype, region, view, ...)
    "case_images.parquet",       # links images -> case_id / article_id
    "cases.parquet",             # clinical case text, patient age, gender
    "abstracts.parquet",         # article abstracts
    "metadata.parquet",          # article title, journal, year, authors, mesh terms, doi, ...
    "data_dictionary.csv",       # column-by-column documentation of every file above
]

for fname in META_FILES:
    download(f"{ZENODO_RECORD}/{fname}?download=1", DATA_DIR / fname)


already have captions_and_labels.csv
already have case_images.parquet
already have cases.parquet
already have abstracts.parquet
already have metadata.parquet
already have data_dictionary.csv


## Load the tables into pandas

In [5]:
captions_df    = pd.read_csv(DATA_DIR / "captions_and_labels.csv")
case_images_df = pd.read_parquet(DATA_DIR / "case_images.parquet")
cases_df       = pd.read_parquet(DATA_DIR / "cases.parquet")
abstracts_df   = pd.read_parquet(DATA_DIR / "abstracts.parquet")
metadata_df    = pd.read_parquet(DATA_DIR / "metadata.parquet")
data_dictionary_df = pd.read_csv(DATA_DIR / "data_dictionary.csv")

print(f"{len(captions_df):,} image records (captions_and_labels.csv)")
print(f"{len(case_images_df):,} case-image links (case_images.parquet)")
print(f"{len(cases_df):,} clinical cases (cases.parquet)")
print(f"{len(abstracts_df):,} article abstracts (abstracts.parquet)")
print(f"{len(metadata_df):,} articles (metadata.parquet)")


/tmp/ipykernel_20081/3405431455.py:1: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  captions_df    = pd.read_csv(DATA_DIR / "captions_and_labels.csv")


139,254 image records (captions_and_labels.csv)
76,137 case-image links (case_images.parquet)
76,137 clinical cases (cases.parquet)
76,137 article abstracts (abstracts.parquet)
76,137 articles (metadata.parquet)


### Flatten any nested tables

Some dataset versions store one row per **article** with the per-image /
per-case records nested inside a single column (as lists, numpy arrays, or
JSON strings) instead of one flat row per record. The helper below detects
and unnests any such column, however deep, for every table.


In [6]:
import json
import numpy as np

def _is_nested(v):
    return isinstance(v, (list, tuple, np.ndarray, dict))

def _to_dict(v):
    if isinstance(v, dict):
        return v
    if isinstance(v, str):
        try:
            return json.loads(v)
        except Exception:
            return {}
    try:
        return dict(v)
    except Exception:
        return {}

def flatten_nested_df(df, required_col, df_name, max_depth=5):
    """Repeatedly explode/normalize nested columns in df until
    `required_col` is present as a flat column, or no nested columns remain.
    """
    for _ in range(max_depth):
        if required_col in df.columns:
            break
        for c in df.columns:
            sample = df[c].dropna()
            if len(sample):
                print(f"  [{df_name}] {c}: dtype={df[c].dtype}, sample type={type(sample.iloc[0])}")
        list_cols = [c for c in df.columns if df[c].dropna().apply(_is_nested).any()]
        if not list_cols:
            print(f"[{df_name}] no nested columns detected but '{required_col}' still missing - inspect columns above manually")
            break
        col = list_cols[0]
        exploded = df.explode(col).reset_index(drop=True)
        exploded[col] = exploded[col].apply(_to_dict)
        nested = pd.json_normalize(exploded[col])
        dup_cols = [c for c in nested.columns if c in exploded.columns]
        nested = nested.drop(columns=dup_cols)  # avoid duplicates (e.g. article_id at both levels)
        df = pd.concat([exploded.drop(columns=[col]).reset_index(drop=True), nested], axis=1)
        print(f"[{df_name}] flattened nested '{col}' column -> {list(df.columns)}")
    return df


captions_df    = flatten_nested_df(captions_df, "caption", "captions_df")
case_images_df = flatten_nested_df(case_images_df, "image_id", "case_images_df")
cases_df       = flatten_nested_df(cases_df, "case_id", "cases_df")
abstracts_df   = flatten_nested_df(abstracts_df, "abstract", "abstracts_df")
metadata_df    = flatten_nested_df(metadata_df, "title", "metadata_df")


  [case_images_df] case_images: dtype=object, sample type=<class 'numpy.ndarray'>
  [case_images_df] article_id: dtype=object, sample type=<class 'str'>
[case_images_df] flattened nested 'case_images' column -> ['article_id', 'case_id', 'case_image_list']
  [case_images_df] article_id: dtype=object, sample type=<class 'str'>
  [case_images_df] case_id: dtype=object, sample type=<class 'str'>
  [case_images_df] case_image_list: dtype=object, sample type=<class 'numpy.ndarray'>
[case_images_df] flattened nested 'case_image_list' column -> ['article_id', 'case_id', 'caption', 'file', 'image_id', 'tag', 'text_references']
  [cases_df] cases: dtype=object, sample type=<class 'numpy.ndarray'>
  [cases_df] article_id: dtype=object, sample type=<class 'str'>
[cases_df] flattened nested 'cases' column -> ['article_id', 'age', 'case_id', 'case_text', 'gender']
  [metadata_df] article_id: dtype=object, sample type=<class 'str'>
  [metadata_df] article_metadata: dtype=object, sample type=<class 'd

In [7]:
# column reference for every table
data_dictionary_df


,file,field,explanation
0,captions_and_labels.csv,file_id,Primary key for each row. Each row contains on...
1,captions_and_labels.csv,file,Name of the image file. The file path can be d...
2,captions_and_labels.csv,main_image,Id from the original image (it corresponds to ...
3,captions_and_labels.csv,image_component,It is 'undivided' if the source image was not ...
4,captions_and_labels.csv,patient_id,"Id of the patient, created combining the PMC o..."
5,captions_and_labels.csv,license,License of the article. The possible values ar...
6,captions_and_labels.csv,file_size,Size of the corresponding image (in bytes).
7,captions_and_labels.csv,caption,It is the caption that corresponds to the imag...
8,captions_and_labels.csv,case_substring,Part of the clinical case that references the ...
9,captions_and_labels.csv,image_type,Multiclass classification column with seven po...


In [8]:
# inspect actual column names on disk — dataset schemas can change between
# versions, so we check this before merging anything below.
for name, df in [
    ("captions_df", captions_df),
    ("case_images_df", case_images_df),
    ("cases_df", cases_df),
    ("abstracts_df", abstracts_df),
    ("metadata_df", metadata_df),
]:
    print(f"{name}: {list(df.columns)}\n")


captions_df: ['file_id', 'file', 'main_image', 'image_component', 'patient_id', 'license', 'file_size', 'caption', 'case_substring', 'image_type', 'image_subtype', 'radiology_region', 'radiology_region_granular', 'radiology_view', 'ml_labels_for_supervised_classification', 'gt_labels_for_semisupervised_classification', 'main_image_link']

case_images_df: ['article_id', 'case_id', 'caption', 'file', 'image_id', 'tag', 'text_references']

cases_df: ['article_id', 'age', 'case_id', 'case_text', 'gender']

abstracts_df: ['abstract', 'article_id']

metadata_df: ['article_id']



In [9]:
captions_df.head(3)


,file_id,file,main_image,image_component,patient_id,license,file_size,caption,case_substring,image_type,image_subtype,radiology_region,radiology_region_granular,radiology_view,ml_labels_for_supervised_classification,gt_labels_for_semisupervised_classification,main_image_link
0,file_0000000,PMC10000323_jbsr-107-1-3012-g3_undivided_1_1.webp,PMC10000323_01_jbsr-107-1-3012-g3.jpg,undivided,PMC10000323_01,CC BY,105470,Pathological result.,['Figure 3'],pathology,h&e,NaN,NaN,NaN,"['pathology', 'h&e']",[],NaN
1,file_0000001,PMC10000728_fmed-09-985235-g001_A_1_3.webp,PMC10000728_01_fmed-09-985235-g001.jpg,a,PMC10000728_01,CC BY,25364,Intraoperative exploration revealed a teratoma...,['Figure 1A'],medical_photograph,other_medical_photograph,NaN,NaN,NaN,"['medical_photograph', 'other_medical_photogra...",[],NaN
2,file_0000002,PMC10000728_fmed-09-985235-g001_B_2_3.webp,PMC10000728_01_fmed-09-985235-g001.jpg,b,PMC10000728_01,CC BY,27484,The teratoma was disconnected from the posteri...,['Figure 1B'],medical_photograph,other_medical_photograph,NaN,NaN,NaN,"['medical_photograph', 'other_medical_photogra...",[],NaN


In [10]:
cases_df.head(3)


,article_id,age,case_id,case_text,gender
0,PMC3738355,53.0,PMC3738355_01,A 53-year-old woman presented with a 10-year h...,Female
1,PMC5015624,69.0,PMC5015624_01,A 69-year-old Caucasian female with coronary a...,Female
2,PMC6381877,60.0,PMC6381877_01,A 60-year-old male smoker presented with persi...,Male


In [11]:
metadata_df.head(3)


,article_id
0,PMC3738355
1,PMC3738355
2,PMC3738355


## Download image shard(s)

Per `data_dictionary.csv`, an image file's on-disk path is derived from its
own name: `f"{file[:4]}/{file[:5]}/{file}"`. The leading 4 characters (e.g.
`PMC2`) indicate which zip shard (`PMC2.zip`) contains it, so we download and
extract only the shard(s) listed in `SHARDS` above.


In [12]:
# if DOWNLOAD_IMAGES:
#     IMG_DIR.mkdir(exist_ok=True, parents=True)
#     for shard in SHARDS:
#         zip_path = download(f"{ZENODO_RECORD}/{shard}.zip?download=1", DATA_DIR / f"{shard}.zip")

#         marker = IMG_DIR / f".{shard}_extracted"
#         if marker.exists():
#             print(f"{shard} already extracted")
#             continue

#         print(f"extracting {zip_path.name} ...")
#         with zipfile.ZipFile(zip_path) as zf:
#             zf.extractall(IMG_DIR)
#         marker.touch()
#     print("all shards downloaded and extracted")
# else:
#     print("DOWNLOAD_IMAGES is False - skipping image download")


In [13]:
# index every extracted image: filename -> path on disk
# (image format has varied across dataset versions - jpg, jpeg, png, webp -
#  so we index all common formats rather than assuming one extension)
IMAGE_EXTENSIONS = ["jpg", "jpeg", "png", "webp"]

image_index = {}
if DOWNLOAD_IMAGES:
    for ext in IMAGE_EXTENSIONS:
        for p in IMG_DIR.rglob(f"*.{ext}"):
            image_index[p.name] = p

print(f"{len(image_index):,} images available on disk")
if image_index:
    from collections import Counter
    ext_counts = Counter(p.suffix.lower() for p in image_index.values())
    print("by extension:", dict(ext_counts))

def get_image_path(file_name: str):
    """Look up the local path of an image file by its name from captions_and_labels.csv['file']."""
    return image_index.get(file_name)


139,254 images available on disk
by extension: {'.webp': 139254}


## Build one merged table

One row per image, joined with:
- its ground-truth caption and visual labels (`captions_and_labels.csv`)
- the case/article it belongs to (`case_images.parquet`)
- the full clinical case text, patient age & gender (`cases.parquet`)
- the article abstract (`abstracts.parquet`)
- article metadata: title, journal, year, authors, doi, mesh terms
  (`metadata.parquet`)
- the image's local file path, if downloaded


In [14]:
def pick_col(df, candidates, df_name):
    """Return the first candidate column name that actually exists in df.
    Raises a clear error (listing the real columns) if none match, since
    dataset schemas can change between versions.
    """
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"None of {candidates} found in {df_name}. "
        f"Actual columns: {list(df.columns)}"
    )

def safe_select(df, candidates, df_name):
    """Keep only the columns from `candidates` that actually exist in df,
    warning about (and skipping) any that don't."""
    present = [c for c in candidates if c in df.columns]
    missing = [c for c in candidates if c not in df.columns]
    if missing:
        print(f"note: {df_name} is missing {missing} (skipping) — actual columns: {list(df.columns)}")
    return df[present]


# join keys — matched dynamically in case column names shift between dataset versions
main_image_key = pick_col(captions_df, ["main_image", "image_id"], "captions_df")
ci_image_key   = pick_col(case_images_df, ["image_id", "main_image", "id"], "case_images_df")
ci_case_key    = pick_col(case_images_df, ["case_id"], "case_images_df")
cases_case_key = pick_col(cases_df, ["case_id"], "cases_df")

merged_df = (
    captions_df
    .merge(
        safe_select(case_images_df, [ci_image_key, ci_case_key, "article_id"], "case_images_df"),
        left_on=main_image_key, right_on=ci_image_key, how="left",
    )
    .merge(
        safe_select(cases_df, [cases_case_key, "case_text", "age", "gender"], "cases_df"),
        left_on=ci_case_key, right_on=cases_case_key, how="left",
    )
    .merge(
        safe_select(abstracts_df, ["article_id", "abstract"], "abstracts_df"),
        on="article_id", how="left",
    )
    .merge(
        safe_select(
            metadata_df,
            ["article_id", "title", "authors", "journal", "year", "doi",
             "pmid", "pmcid", "mesh_terms", "major_mesh_terms", "keywords", "link"],
            "metadata_df",
        ),
        on="article_id", how="left",
    )
)

merged_df["local_path"] = merged_df["file"].apply(get_image_path) if DOWNLOAD_IMAGES else None

print(f"{len(merged_df):,} rows (one per image) in merged_df")
print(f"{len(merged_df.columns)} columns: {list(merged_df.columns)}")
if DOWNLOAD_IMAGES:
    print(f"{merged_df['local_path'].notna().sum():,} rows have a local image file on disk")
merged_df.head(3)


note: metadata_df is missing ['title', 'authors', 'journal', 'year', 'doi', 'pmid', 'pmcid', 'mesh_terms', 'major_mesh_terms', 'keywords', 'link'] (skipping) — actual columns: ['article_id']
1,989,750 rows (one per image) in merged_df
25 columns: ['file_id', 'file', 'main_image', 'image_component', 'patient_id', 'license', 'file_size', 'caption', 'case_substring', 'image_type', 'image_subtype', 'radiology_region', 'radiology_region_granular', 'radiology_view', 'ml_labels_for_supervised_classification', 'gt_labels_for_semisupervised_classification', 'main_image_link', 'image_id', 'case_id', 'article_id', 'case_text', 'age', 'gender', 'abstract', 'local_path']
1,989,750 rows have a local image file on disk


,file_id,file,main_image,image_component,patient_id,license,file_size,caption,case_substring,image_type,...,gt_labels_for_semisupervised_classification,main_image_link,image_id,case_id,article_id,case_text,age,gender,abstract,local_path
0,file_0000000,PMC10000323_jbsr-107-1-3012-g3_undivided_1_1.webp,PMC10000323_01_jbsr-107-1-3012-g3.jpg,undivided,PMC10000323_01,CC BY,105470,Pathological result.,['Figure 3'],pathology,...,[],NaN,PMC10000323_01_jbsr-107-1-3012-g3.jpg,PMC10000323_01,PMC10000323,A 45-year-old man with pain and numbness in th...,45.0,Male,Teaching Point: Giant cell tumor of bone may s...,multicare_data/images/PMC1/PMC10/PMC10000323_j...
1,file_0000000,PMC10000323_jbsr-107-1-3012-g3_undivided_1_1.webp,PMC10000323_01_jbsr-107-1-3012-g3.jpg,undivided,PMC10000323_01,CC BY,105470,Pathological result.,['Figure 3'],pathology,...,[],NaN,PMC10000323_01_jbsr-107-1-3012-g3.jpg,PMC10000323_01,PMC10000323,A 45-year-old man with pain and numbness in th...,45.0,Male,Teaching Point: Giant cell tumor of bone may s...,multicare_data/images/PMC1/PMC10/PMC10000323_j...
2,file_0000000,PMC10000323_jbsr-107-1-3012-g3_undivided_1_1.webp,PMC10000323_01_jbsr-107-1-3012-g3.jpg,undivided,PMC10000323_01,CC BY,105470,Pathological result.,['Figure 3'],pathology,...,[],NaN,PMC10000323_01_jbsr-107-1-3012-g3.jpg,PMC10000323_01,PMC10000323,A 45-year-old man with pain and numbness in th...,45.0,Male,Teaching Point: Giant cell tumor of bone may s...,multicare_data/images/PMC1/PMC10/PMC10000323_j...


In [18]:
# Stream the merge + write in row-chunks over captions_df directly, instead
# of building the full merged_df in memory first.
import gc
import pyarrow as pa
import pyarrow.parquet as pq

CHUNK_SIZE = 3_000
csv_path = DATA_DIR / "merged_dataset.csv"
parquet_path = DATA_DIR / "merged_dataset.parquet"

# small lookup tables -> dicts (cheap: one entry per case/article, not per image)
# drop_duplicates first: set_index(...).to_dict("index") requires a unique index
ci_lookup = (
    case_images_df.drop_duplicates(subset=ci_image_key, keep="first")
    .set_index(ci_image_key)[[ci_case_key, "article_id"]]
    .to_dict("index")
)
case_lookup = (
    cases_df.drop_duplicates(subset=cases_case_key, keep="first")
    .set_index(cases_case_key)[["case_text", "age", "gender"]]
    .to_dict("index")
)
abstract_lookup = (
    abstracts_df.drop_duplicates(subset="article_id", keep="first")
    .set_index("article_id")["abstract"]
    .to_dict()
)
meta_cols = [c for c in ["title", "authors", "journal", "year", "doi", "pmid",
                          "pmcid", "mesh_terms", "major_mesh_terms", "keywords", "link"]
             if c in metadata_df.columns]
meta_lookup = (
    metadata_df.drop_duplicates(subset="article_id", keep="first")
    .set_index("article_id")[meta_cols]
    .to_dict("index")
)

def build_chunk(rows: pd.DataFrame) -> pd.DataFrame:
    records = []
    for r in rows.to_dict("records"):
        ci = ci_lookup.get(r[main_image_key], {})
        case_id = ci.get(ci_case_key)
        article_id = ci.get("article_id")
        case_info = case_lookup.get(case_id, {})
        meta_info = meta_lookup.get(article_id, {})
        records.append({
            **r,
            ci_case_key: case_id,
            "article_id": article_id,
            "case_text": case_info.get("case_text"),
            "age": case_info.get("age"),
            "gender": case_info.get("gender"),
            "abstract": abstract_lookup.get(article_id),
            **{c: meta_info.get(c) for c in meta_cols},
            "local_path": str(get_image_path(r["file"])) if DOWNLOAD_IMAGES else None,
        })
    return pd.DataFrame(records)

# columns that should stay numeric — every other column is forced to
# pandas' nullable "string" dtype, which maps to a fixed pyarrow string
# type even when an entire chunk's column is null. Plain object/None gets
# inferred as pyarrow's `null` type instead, which breaks the
# ParquetWriter's fixed schema check across chunks.
NUMERIC_COLS = {"file_size", "age", "year", "case_amount"}

def coerce_chunk_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if col in NUMERIC_COLS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            df[col] = df[col].apply(lambda v: str(v) if pd.notna(v) else pd.NA).astype("string")
    return df

writer = None
n_written = 0
try:
    for start in range(0, len(captions_df), CHUNK_SIZE):
        rows = captions_df.iloc[start:start + CHUNK_SIZE]
        chunk = coerce_chunk_dtypes(build_chunk(rows))

        chunk.to_csv(
            csv_path, index=False,
            mode="w" if start == 0 else "a", header=(start == 0),
        )

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(parquet_path, table.schema)
        writer.write_table(table)

        n_written += len(chunk)
        del rows, chunk, table
        gc.collect()
finally:
    if writer is not None:
        writer.close()

print(f"saved -> {parquet_path}  ({n_written:,} rows)")
print(f"saved -> {csv_path}")

saved -> multicare_data/merged_dataset.parquet  (139,254 rows)
saved -> multicare_data/merged_dataset.csv


## Upload to the Hugging Face Hub

Pushes the merged table **with the images embedded** to a Hugging Face
Hub dataset repo, so both the table and every image live in one place at
`https://huggingface.co/datasets/<REPO_ID>`.

> ⚠️ **Licensing reminder before you upload:** the dataset overall is
> CC BY-NC-SA, but individual rows may carry a *less* restrictive license
> (see the `license` column). Uploading publicly redistributes that content
> under those terms — set `PRIVATE = True` below unless you're sure that's
> what you want, and keep the `license` column in the uploaded table so
> downstream users can see per-row licensing.


In [1]:
%pip install -q datasets huggingface_hub


In [4]:
import os
from getpass import getpass
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter your Hugging Face token (needs write access): ")

login(token=HF_TOKEN)


Enter your Hugging Face token (needs write access): ··········


### Configuration

In [5]:
REPO_ID = "MohamedAhmedAE/multicare-subset"   # <-- change this to <your-hf-username>/<dataset-name>
PRIVATE = False                                # set False to make the repo public


### Prepare the table for upload

Only rows with a downloaded image are included (`local_path` not null).
The `datasets` library needs the image column to hold actual file paths
(it will embed the image bytes automatically), and any list/dict-valued
metadata columns (e.g. `authors`, `mesh_terms`) get serialized to JSON
strings so the table has a clean, uniform schema.


In [9]:
import pandas as pd


merged_df = pd.read_parquet("/content/multicare_data/merged_dataset.parquet")
print(f"{len(merged_df):,} rows ready to upload")
merged_df.head(3)

139,254 rows ready to upload


,file_id,file,main_image,image_component,patient_id,license,file_size,caption,case_substring,image_type,...,ml_labels_for_supervised_classification,gt_labels_for_semisupervised_classification,main_image_link,case_id,article_id,case_text,age,gender,abstract,local_path
0,file_0000000,PMC10000323_jbsr-107-1-3012-g3_undivided_1_1.webp,PMC10000323_01_jbsr-107-1-3012-g3.jpg,undivided,PMC10000323_01,CC BY,105470,Pathological result.,['Figure 3'],pathology,...,"['pathology', 'h&e']",[],<NA>,PMC10000323_01,PMC10000323,A 45-year-old man with pain and numbness in th...,45.0,Male,Teaching Point: Giant cell tumor of bone may s...,multicare_data/images/PMC1/PMC10/PMC10000323_j...
1,file_0000001,PMC10000728_fmed-09-985235-g001_A_1_3.webp,PMC10000728_01_fmed-09-985235-g001.jpg,a,PMC10000728_01,CC BY,25364,Intraoperative exploration revealed a teratoma...,['Figure 1A'],medical_photograph,...,"['medical_photograph', 'other_medical_photogra...",[],<NA>,PMC10000728_01,PMC10000728,A 71-year-old women was admitted to the hospit...,71.0,Female,"Teratomas often occur in the gonads, while Ext...",multicare_data/images/PMC1/PMC10/PMC10000728_f...
2,file_0000002,PMC10000728_fmed-09-985235-g001_B_2_3.webp,PMC10000728_01_fmed-09-985235-g001.jpg,b,PMC10000728_01,CC BY,27484,The teratoma was disconnected from the posteri...,['Figure 1B'],medical_photograph,...,"['medical_photograph', 'other_medical_photogra...",[],<NA>,PMC10000728_01,PMC10000728,A 71-year-old women was admitted to the hospit...,71.0,Female,"Teratomas often occur in the gonads, while Ext...",multicare_data/images/PMC1/PMC10/PMC10000728_f...


In [10]:
def stringify_complex_columns(df):
    """Convert any list/dict/array-valued columns to JSON strings so the
    table has a flat, uniform schema the `datasets` library can infer."""
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object:
            sample = df[c].dropna()
            if len(sample) and isinstance(sample.iloc[0], (list, dict, tuple, np.ndarray)):
                df[c] = df[c].apply(
                    lambda v: json.dumps(v, default=str) if isinstance(v, (list, dict, tuple, np.ndarray)) else v
                )
    return df


hf_df = merged_df[merged_df["local_path"].notna()].copy()
hf_df = stringify_complex_columns(hf_df)
hf_df["image"] = hf_df["local_path"].astype(str)
hf_df = hf_df.drop(columns=["local_path"])

print(f"{len(hf_df):,} rows ready to upload")
hf_df.head(3)


139,254 rows ready to upload


,file_id,file,main_image,image_component,patient_id,license,file_size,caption,case_substring,image_type,...,ml_labels_for_supervised_classification,gt_labels_for_semisupervised_classification,main_image_link,case_id,article_id,case_text,age,gender,abstract,image
0,file_0000000,PMC10000323_jbsr-107-1-3012-g3_undivided_1_1.webp,PMC10000323_01_jbsr-107-1-3012-g3.jpg,undivided,PMC10000323_01,CC BY,105470,Pathological result.,['Figure 3'],pathology,...,"['pathology', 'h&e']",[],<NA>,PMC10000323_01,PMC10000323,A 45-year-old man with pain and numbness in th...,45.0,Male,Teaching Point: Giant cell tumor of bone may s...,multicare_data/images/PMC1/PMC10/PMC10000323_j...
1,file_0000001,PMC10000728_fmed-09-985235-g001_A_1_3.webp,PMC10000728_01_fmed-09-985235-g001.jpg,a,PMC10000728_01,CC BY,25364,Intraoperative exploration revealed a teratoma...,['Figure 1A'],medical_photograph,...,"['medical_photograph', 'other_medical_photogra...",[],<NA>,PMC10000728_01,PMC10000728,A 71-year-old women was admitted to the hospit...,71.0,Female,"Teratomas often occur in the gonads, while Ext...",multicare_data/images/PMC1/PMC10/PMC10000728_f...
2,file_0000002,PMC10000728_fmed-09-985235-g001_B_2_3.webp,PMC10000728_01_fmed-09-985235-g001.jpg,b,PMC10000728_01,CC BY,27484,The teratoma was disconnected from the posteri...,['Figure 1B'],medical_photograph,...,"['medical_photograph', 'other_medical_photogra...",[],<NA>,PMC10000728_01,PMC10000728,A 71-year-old women was admitted to the hospit...,71.0,Female,"Teratomas often occur in the gonads, while Ext...",multicare_data/images/PMC1/PMC10/PMC10000728_f...


### Build the Hugging Face `Dataset` and push it

For a first test, upload a small slice (e.g. `hf_df.head(50)`) before
committing to pushing the full table — for 100k+ images this step can take
a while and `push_to_hub` will auto-shard the data into multiple parquet
files.


In [11]:
from datasets import Dataset, Image as HFImage

ds = Dataset.from_pandas(hf_df, preserve_index=False)
ds = ds.cast_column("image", HFImage())
ds


Dataset({
    features: ['file_id', 'file', 'main_image', 'image_component', 'patient_id', 'license', 'file_size', 'caption', 'case_substring', 'image_type', 'image_subtype', 'radiology_region', 'radiology_region_granular', 'radiology_view', 'ml_labels_for_supervised_classification', 'gt_labels_for_semisupervised_classification', 'main_image_link', 'case_id', 'article_id', 'case_text', 'age', 'gender', 'abstract', 'image'],
    num_rows: 139254
})

In [12]:
ds.push_to_hub(REPO_ID, private=PRIVATE)
print(f"pushed to https://huggingface.co/datasets/{REPO_ID}")


Uploading the dataset shards:   0%|          | 0/7 [00:00<?, ? shards/s]

Map:   0%|          | 0/19894 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/199 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/199 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   0%|          | 1.54MB /  380MB            

pushed to https://huggingface.co/datasets/MohamedAhmedAE/multicare-subset


### (Optional) Add a dataset card with license & citation info

In [13]:
from huggingface_hub import DatasetCard, DatasetCardData

card_data = DatasetCardData(
    license="cc-by-nc-sa-4.0",
    language="en",
    task_categories=["image-to-text", "visual-question-answering", "image-classification"],
    pretty_name="MultiCaRe (subset)",
)

card_text = f"""
# MultiCaRe (subset)

A subset of the [MultiCaRe](https://zenodo.org/records/20416562) open-source
clinical case dataset, re-packaged for the Hugging Face Hub with images
embedded alongside captions, clinical case text, and article metadata.

- **Source**: https://zenodo.org/records/20416562 (DOI: 10.5281/zenodo.20416562)
- **Original paper**: https://doi.org/10.3390/data10080123
- **License**: the dataset as a whole is CC BY-NC-SA 4.0. Individual rows may
  carry a *less* restrictive license (CC BY, CC BY-NC, CC0) recorded in the
  `license` column — check it before reuse, especially anything commercial.
- **Rows in this subset**: {len(hf_df):,}

## Citation

```
Nievas Offidani, M., Roffet, F., González Galtier, M. C., Massiris, M., & Delrieux, C. (2025).
An Open-Source Clinical Case Dataset for Medical Image Classification and Multimodal AI
Applications. Data, 10(8), 123. https://doi.org/10.3390/data10080123

Nievas Offidani, M. (2025). MultiCaRe: An open-source clinical case dataset for medical image
classification and multimodal AI applications (version 3) [Data set].
Zenodo. https://doi.org/10.5281/zenodo.20416562
```
"""

card = DatasetCard(card_text)
card.data = card_data
card.push_to_hub(REPO_ID)
print(f"dataset card pushed to https://huggingface.co/datasets/{REPO_ID}")


Repo card metadata block was not found. Setting CardData to empty.


dataset card pushed to https://huggingface.co/datasets/MohamedAhmedAE/multicare-subset


## Notes

- **Full dataset:** to fetch every image, set `SHARDS` to all nine shard
  names (`PMC1`..`PMC9`, ~3 GB total) or `DOWNLOAD_IMAGES = False` if you
  only need the metadata/captions/case text.
- **Filtered subsets:** for filtering across the *entire* dataset by
  demographics, keywords, or image labels without downloading everything
  manually, the maintainers provide the `multiversity` library:
  ```bash
  pip install multiversity
  ```
  ```python
  from multiversity.multicare_dataset import MedicalDatasetCreator
  mdc = MedicalDatasetCreator(directory="medical_datasets")
  filters = [
      {"field": "case_strings", "string_list": ["tumor", "cancer"], "operator": "any"},
      {"field": "label", "string_list": ["mri", "head"]},
  ]
  mdc.create_dataset(dataset_name="brain_tumor_subset", filter_list=filters, dataset_type="multimodal")
  ```
- **Licensing:** the dataset overall is CC BY-NC-SA, but each image/article
  carries its own license in the `license` column (CC BY, CC BY-NC,
  CC BY-NC-SA, etc.) — check before reuse.
- **Citation:**
  ```
  Nievas Offidani, M., Roffet, F., González Galtier, M. C., Massiris, M., & Delrieux, C. (2025).
  An Open-Source Clinical Case Dataset for Medical Image Classification and Multimodal AI
  Applications. Data, 10(8), 123. https://doi.org/10.3390/data10080123

  Nievas Offidani, M. (2025). MultiCaRe: An open-source clinical case dataset for medical image
  classification and multimodal AI applications (version 3) [Data set].
  Zenodo. https://doi.org/10.5281/zenodo.20416562
  ```
